In [71]:
# Predicting effects of variance using trained models

import anndata as ad
import pandas as pd
import numpy as np
import os
import torch

import glob
import grelu
import grelu.lightning

import sys
sys.path.append("../gReLU/src")

from grelu.lightning import LightningModel
import anndata as ad

In [72]:
experiment_dir = "final_atac_model_2"

try:
    checkpoint_path = glob.glob(f"{experiment_dir}/**/*.ckpt", recursive=True)[0]
except IndexError:
    raise FileNotFoundError(f"Could not find a .ckpt file in {experiment_dir}")

print(f"Found checkpoint file: {checkpoint_path}")

# Load the model from the checkpoint file
model = grelu.lightning.LightningModel.load_from_checkpoint(checkpoint_path)
model.eval()

print("Model loaded successfully.")
print(model)

Found checkpoint file: final_atac_model_2/2025_20_08_19_27/version_0/checkpoints/last.ckpt
Model loaded successfully.
LightningModel(
  (model): ConvModel(
    (embedding): ConvTrunk(
      (conv_tower): ConvTower(
        (blocks): ModuleList(
          (0): Stem(
            (conv): Conv1d(4, 64, kernel_size=(15,), stride=(1,), padding=same)
            (act): Activation(
              (layer): GELU(approximate='none')
            )
            (pool): Pool(
              (layer): Identity()
            )
          )
          (1): ConvBlock(
            (norm): Norm(
              (layer): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            )
            (conv): Conv1d(64, 256, kernel_size=(9,), stride=(1,), padding=same)
            (act): Activation(
              (layer): GELU(approximate='none')
            )
            (pool): Pool(
              (layer): AvgPool1d(kernel_size=(4,), stride=(4,), padding=(0,))
            )
            (

In [73]:
tasks = pd.DataFrame(model.data_params['tasks'])
tasks.head(3)

,name,task_name
0,zldrnai_15to2_atac,zldrnai_15to2_atac
1,zldrnai_1to15_atac,zldrnai_1to15_atac
2,gd7_1to15_atac,gd7_1to15_atac


In [74]:
# Load significant regions

regions_bed = "final_t_test_results_qnorm_q005/ATAC_UNION_q_significant_regions.bed"
regions_df = pd.read_csv(regions_bed, sep="\t", header=None, names=["chrom", "start", "end"])
print(f"Loaded {len(regions_df)} significant regions.")

Loaded 3436 significant regions.


In [75]:
all_task_names = [
    "orer_gaf_mbt_nexus", "zldrnai_15to2_atac", "orer_dl_mbt_nexus",
    "zldrnai_1to15_atac", "orer_zld_mbt_nexus", "gd7_1to15_atac",
    "orer_h3k27ac_mbt_seq", "gd7_15to2_atac", "orer_2to25_atac",
    "orer_cad_mbt_nexus", "zldrnai_25to3_atac", "gd7_2to25_atac",
    "orer_wce_mbt_seq", "orer_1to15_atac", "orer_twi_mbt_nexus",
    "cic6_2to25_atac", "orer_25to3_atac", "zldrnai_2to25_atac",
    "orer_bcd_mbt_nexus", "cic6_25to3_atac", "orer_15to2_atac",
    "gd7_25to3_atac", "orer_h3k4me1_mbt_seq", "orer_mbt_100u_mnase"
]

all_tasks_series = pd.Series(all_task_names)
is_atac_mask = all_tasks_series.str.contains("atac")
atac_task_names = all_tasks_series[is_atac_mask].tolist()

print(len(atac_task_names), "ATAC tasks")
print(atac_task_names)


14 ATAC tasks
['zldrnai_15to2_atac', 'zldrnai_1to15_atac', 'gd7_1to15_atac', 'gd7_15to2_atac', 'orer_2to25_atac', 'zldrnai_25to3_atac', 'gd7_2to25_atac', 'orer_1to15_atac', 'cic6_2to25_atac', 'orer_25to3_atac', 'zldrnai_2to25_atac', 'cic6_25to3_atac', 'orer_15to2_atac', 'gd7_25to3_atac']


In [76]:
comparisons = [
("timepoint_orer_1to15_vs_15to2"),
("timepoint_orer_15to2_vs_2to25"),
("timepoint_orer_2to25_vs_25to3"),
("timepoint_cic6_2to25_vs_25to3"),
("timepoint_zldrnai_1to15_vs_15to2"),
("timepoint_zldrnai_15to2_vs_2to25"),
("timepoint_zldrnai_2to25_vs_25to3"),
("timepoint_gd7_1to15_vs_15to2"),
("timepoint_gd7_15to2_vs_2to25"),
("timepoint_gd7_2to25_vs_25to3"),
("mutant_zldrnai_vs_orer_1to15"),
("mutant_gd7_vs_orer_1to15"),
("mutant_zldrnai_vs_orer_15to2"),
("mutant_gd7_vs_orer_15to2"),
("mutant_cic6_vs_orer_2to25"),
("mutant_zldrnai_vs_orer_2to25"),
("mutant_gd7_vs_orer_2to25"),
("mutant_cic6_vs_orer_25to3"),
("mutant_zldrnai_vs_orer_25to3"),
("mutant_gd7_vs_orer_25to3"),
]

In [ ]:
# 3) Function to build an AnnData with only the two tasks (conditions) we want

def make_ad_for_tasks_and_regions(task_a, task_b, regions_bed):
    obs = pd.DataFrame({"task_name":[task_a, task_b]}).set_index("task_name")
    var = regions_bed.copy()
    var.index = var["chrom"] + "_" + var["start"].astype(str) + "_" + var["end"].astype(str)
    # X is not used for prediction; shape must be (n_tasks, n_regions)
    X = np.zeros((len(obs), len(var)), dtype=np.float32)
    adata = ad.AnnData(X=X, obs=obs, var=var)
    return adata

In [ ]:
# 4) A helper to get predictions (tasks x regions) for a given comparison
def predict_for_comparison(task_a, task_b, regions_bed, trained_task_names,
                           genome="dm6", seq_len=2000, batch_size=128, devices=[0], num_workers=4):
    # Build tiny AnnData (pred API still outputs all heads)
    obs = pd.DataFrame({"task_name":[task_a, task_b]}).set_index("task_name")
    var = regions_bed.copy()
    var.index = var["chrom"] + "_" + var["start"].astype(str) + "_" + var["end"].astype(str)
    X = np.zeros((len(obs), len(var)), dtype=np.float32)
    ad_small = ad.AnnData(X=X, obs=obs, var=var)

    dataset = grelu.data.dataset.AnnDataSeqDataset(
        ad_small, genome=genome, seq_len=seq_len, rc=False, max_seq_shift=0
    )

    preds = model.predict_on_dataset(
        dataset=dataset,
        devices=devices,
        num_workers=num_workers,
        batch_size=batch_size,
    )

    # to numpy 
    if isinstance(preds, list):
        preds = np.concatenate([
            (p.detach().cpu().numpy() if hasattr(p, "detach") else np.asarray(p))
            for p in preds
        ], axis=0)
    elif hasattr(preds, "detach"):
        preds = preds.detach().cpu().numpy()
    else:
        preds = np.asarray(preds)

    # squeeze/transpose to (n_tasks_total, n_regions)
    if preds.ndim == 3 and preds.shape[-1] == 1:
        preds = preds[..., 0]                # (n_regions, n_tasks_total)
    if preds.shape[0] == len(var):          # (n_regions, n_tasks_total) -> transpose
        preds = preds.T                      # (n_tasks_total, n_regions)
    # sanity check
    if preds.shape[0] != len(trained_task_names):
        raise ValueError(f"Expected {len(trained_task_names)} tasks, got {preds.shape[0]}")

    region_index = list(var.index)
    df_all = pd.DataFrame(preds, index=trained_task_names, columns=region_index)
    df_two = df_all.loc[[task_a, task_b]]
    return df_two


In [79]:
# 5) Run for one comparison
task_a, task_b = "orer_1to15_atac", "gd7_1to15_atac"
pred_df = predict_for_comparison(task_a, task_b, regions_df, trained_task_names=atac_task_names)

# Δpred per region
delta_pred = pred_df.loc[task_b] - pred_df.loc[task_a]
delta_pred.name = f"{task_b}__minus__{task_a}"


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting DataLoader 0: 100%|██████████| 27/27 [00:00<00:00, 42.02it/s]


In [80]:
# 6) Compute model effect (Δpred = task_b - task_a) per region
delta_pred = pred_df.loc[task_b] - pred_df.loc[task_a]
delta_pred.name = f"{task_b}__minus__{task_a}"

In [81]:
print(pred_df.shape)
print("Index (tasks):", pred_df.index.tolist())
print("Columns (regions):", list(pred_df.columns[:5]))  # first 5 region IDs

print(delta_pred.shape)
print(delta_pred.head())   # first 5 values of Δpred


(2, 3436)
Index (tasks): ['orer_1to15_atac', 'gd7_1to15_atac']
Columns (regions): ['chr2L_1326_1526', 'chr2L_5726_5926', 'chr2L_8957_9157', 'chr2L_27181_27381', 'chr2L_27381_27581']
(3436,)
chr2L_1326_1526      34.057869
chr2L_5726_5926      -4.754646
chr2L_8957_9157     -21.443043
chr2L_27181_27381   -29.949547
chr2L_27381_27581   -20.183247
Name: gd7_1to15_atac__minus__orer_1to15_atac, dtype: float32


In [82]:
# Add delta_pred back to the region DataFrame
results_df = regions_df.copy()
results_df["region_id"] = results_df["chrom"] + "_" + results_df["start"].astype(str) + "_" + results_df["end"].astype(str)
results_df = results_df.set_index("region_id")

# Align delta_pred by region index
results_df["pred_orer_1to15"] = pred_df.loc["orer_1to15_atac"].values
results_df["pred_gd7_1to15"]  = pred_df.loc["gd7_1to15_atac"].values
results_df["delta_pred"]      = delta_pred.values

# Save to CSV
out_path = "predictions_orer_vs_gd7_1to15.csv"
results_df.to_csv(out_path, index=True)

print(f"Saved predictions to {out_path}")
print(results_df.head())


Saved predictions to predictions_orer_vs_gd7_1to15.csv
                   chrom  start    end  pred_orer_1to15  pred_gd7_1to15  \
region_id                                                                 
chr2L_1326_1526    chr2L   1326   1526        17.406002       51.463871   
chr2L_5726_5926    chr2L   5726   5926       101.058556       96.303909   
chr2L_8957_9157    chr2L   8957   9157        71.230316       49.787273   
chr2L_27181_27381  chr2L  27181  27381       131.652023      101.702477   
chr2L_27381_27581  chr2L  27381  27581        67.805817       47.622570   

                   delta_pred  
region_id                      
chr2L_1326_1526     34.057869  
chr2L_5726_5926     -4.754646  
chr2L_8957_9157    -21.443043  
chr2L_27181_27381  -29.949547  
chr2L_27381_27581  -20.183247  
